# AI Workflows vs. AI Agents

**Prerequisites:** `01_foundations`, especially `06_function_calling.ipynb` and `09_build_an_agent.ipynb`

Both a workflow and an agent can call an LLM, call a tool, and run several steps to get an answer — that overlap is exactly why the two get confused. The distinction that actually matters is what happens to a step's output once it comes back: a workflow **expects** it to have a certain shape and moves on regardless; an agent **observes** it, actually reads it, and lets it change what happens next.

**By the end of this notebook you'll have:**
- Defined AI workflows and AI agents by that expect-vs-observe difference, not by how many steps or tools are involved
- Traced the agent loop — Reason → Act → Observe — through code you've already built (`resolve_tools()`/`asend_with_tools()`, notebook 6 and 9)
- Found the exact moment that difference already showed up for real: a tool-call validation error being observed and reasoned about, instead of assumed away
- Seen both laid out side by side: a one-way pipeline of expectations vs. a closed loop of observation feeding back into the next decision

## AI Workflows

- LLMs operate inside a **human-defined pipeline** — the developer decides the sequence of steps before any of them run.
- **Execution path is predetermined by code.** Step 2 runs after step 1 because the code says so, not because step 1's output earned it.
- **Each step expects a particular type of output.** The pipeline is written assuming step 1 returns, say, a category string — it isn't checking whether that assumption held, just consuming it.
- **The LLM performs tasks but does not decide what happens next.** It can produce a sharp classification or a bad one; either way, the same next step runs.
- **Best suited to predictable, well-defined processes** — exactly the cases where "expecting" a fixed shape is a safe bet.

## AI Agents

- **AI agents operate through an agent loop: Reason → Act → Observe.** Reason decides what to do, Act does it, Observe looks at what actually happened before the next Reason step begins.
- **The agent decides what to do next during execution** — nobody wrote the branch for it ahead of time.
- **It observes tool results or the environment before making further decisions** — not *assumes*, **observes**: the actual content of the result is read and can change the next decision.
- **It can adapt its path when results are unexpected or circumstances change** — a workflow can't; an unexpected shape either breaks it or gets silently mishandled.
- **That ability to dynamically control its own execution path is what gives the system agency.**

## Expect vs. observe: the crucial difference, in code you've already run

`execute_tool_call()` (notebook 6) validates a tool's arguments before running it. When they're bad, `asend_with_tools()` (notebook 9) doesn't discard the error — it feeds it back as the tool's own result:

```python
async def run(tool_call):
    try:
        return await asyncio.to_thread(execute_tool_call, tool_call, functions, schemas)
    except Exception as e:
        return f"Error: {e}"
...
for tool_call, result in zip(message.tool_calls, results):
    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
```

That `Error: ...` string lands in `messages` exactly like a real result would. The *next* Reason step — the next call to the model — reads it, sees the actual reason its own arguments failed, and corrects them: notebook 9's `convo2` demo proved this live, `"from_unit": "Fahrenheit"` becoming `"from_unit": "F"` on the very next attempt, with no code anywhere branching on "if arguments invalid, fix unit." That correction only exists because the loop **observed** the error's content.

A workflow *could* handle this same failure — but only if someone anticipated it and wrote a branch for it ahead of time: "if `from_unit` validation fails, retry with a fixed lookup like `{"Fahrenheit": "F", "Celsius": "C"}`." That's still a workflow, still fully predetermined, and it would work fine, right up until an outcome nobody enumerated shows up — a typo the lookup table doesn't cover, a validation error on a field the branch wasn't written for, a new failure mode a future tool introduces. Then it breaks or mishandles it, because the code only has branches for the outcomes someone thought to write down. That's the real boundary: a workflow is a great fit exactly when the process is well-understood enough that its outcomes are few, known, and enumerable in advance — every one of them earns its own branch, and the whole thing runs predictably because there's nothing left to be surprised by. An agent doesn't need that enumeration. Observe reads whatever content `execute_tool_call()` actually produced, and Reason decides fresh from it every time, so a failure nobody anticipated still gets a real shot at recovery instead of falling through every branch that was written for something else.

## Both, side by side

**AI Workflow — a one-way pipeline of expectations:**

```
step 1                     step 2                     step 3
(LLM call)          ──►    (LLM call)          ──►    (code)
expects: a category         expects: plain text        expects: a bool
   string back                    back                     back
```

Every arrow points one direction, decided before anything ran, and each step can carry its own branches for the outcomes its author anticipated. What none of them can do is handle a shape nobody thought to write a branch for — that content just falls through, unhandled, because the pipeline's whole design rests on the outcomes being few and known ahead of time.

**AI Agent — a closed loop of observation:**

```
   ┌────────────────────────────────────────────────┐
   │                                                  │
   ▼                                                  │
 Reason                                               │
 (look at the conversation so far,                    │
  decide what to do)                                  │
   │                                                  │
   ▼                                                  │
  Act                                                 │
 (run the tool it asked for,                          │
  or return its answer)                               │
   │                                                  │
   ▼                                                  │
 Observe                                              │
 (read the real result — success                      │
  or a validation error — instead                     │
  of assuming it)                                     │
   │                                                  │
   └──────────────────────────────────────────────────┘
        feeds straight back into the next Reason
```

Same three words as the bullets above, just drawn as what they are: a workflow's diagram is a straight line of branches sized to a known set of outcomes; an agent's is a loop that keeps closing back on itself, because every trip around actually reads whatever came out of the last one instead of requiring it to have been foreseen.

## What comes next

This chapter builds a runtime general enough to run that loop for real, for more than one pattern. `02_the_agent_loop.ipynb` pulls chapter 1's `asend_with_tools()` apart into four pieces — `Runtime`, `Observe`, `Think`, `Act` (the same cycle as Reason → Act → Observe, just named from the observation rather than the decision) — and puts them back together as ReAct, this time declared as a list instead of hardcoded as control flow. `03_retries_and_reflection.ipynb` gives the loop two ways to notice something went wrong and go again. `04_planning.ipynb` decides the whole sequence up front and runs its independent parts at the same time. `05_sub_agents.ipynb` splits the agent itself. `06_agent_patterns.ipynb` writes eleven named patterns as declared loops over all of it.

## What you built

✓ Defined AI workflows (human-defined pipeline, fixed path, each step expects a shape, the LLM never decides what runs next) against AI agents (a Reason → Act → Observe loop, the agent decides what runs next, observes real results, adapts)

✓ Found the expect-vs-observe distinction in code you already ran, not a hypothetical: `execute_tool_call()`'s validation error becoming the tool's own result, read and corrected by the very next Reason step

✓ Seen both as diagrams: a one-way pipeline of expectations vs. a closed loop of observation feeding back into decision

**Next:** `02_the_agent_loop.ipynb` — taking chapter 1's one agent function apart into four components that can be re-ordered, so the loop's shape becomes data instead of control flow.